In [1]:
## Imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold,  cross_val_score

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.metrics import confusion_matrix, classification_report, average_precision_score


# Load the Dataset

In [2]:
## Load data set

df = pd.read_csv("loan_model_dataset.csv")
print(df.shape)
print(df.head)

(70745, 10)
<bound method NDFrame.head of       assetNumber originationDate  obligorCreditScore  \
0       17A000001      2012-01-01               656.0   
1       17A000002      2012-01-01               672.0   
2       17A000003      2012-01-01               675.0   
3       17A000004      2012-01-01               623.0   
4       17A000005      2012-01-01               687.0   
...           ...             ...                 ...   
70740   17A071270      2016-10-01               750.0   
70741   17A071271      2016-11-01               761.0   
70742   17A071272      2016-10-01               701.0   
70743   17A071273      2016-10-01               730.0   
70744   17A071274      2016-11-01               771.0   

       paymentToIncomePercentage  originalInterestRatePercentage  \
0                       0.080969                          0.0829   
1                       0.172831                          0.0989   
2                       0.063686                          0.0490   
3

# Baseline and Dummy Classifier
- Splits the data into features and target
- Splits data into training and testing sets
- Defines dummy classifier to create a baseline
- Prints baseline confusion matrix and scores

In [3]:
## Split data between features and target
X = df.drop(columns=["everDelinquent", "assetNumber", "originationDate"])
y = df["everDelinquent"]

## Split data into train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=32)

## Define the dummy classifier to get a baseline
## Use most_frequent to always predict 0 (never delinquent)
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)

print(confusion_matrix(y_test, dummy_pred))
print(classification_report(y_test, dummy_pred, zero_division = 0))


[[13597     0]
 [  552     0]]
              precision    recall  f1-score   support

           0       0.96      1.00      0.98     13597
           1       0.00      0.00      0.00       552

    accuracy                           0.96     14149
   macro avg       0.48      0.50      0.49     14149
weighted avg       0.92      0.96      0.94     14149



# Initial Logistic Regression Model
- Creates model pipeline including imputing NaN values and scaling the data
- Includes 5 fold cross validation, training on 4 pieces/testing on the 5th and then rotating
- Fits the model and prints a confusion matrix and recall/accuracy/precision scores

In [4]:
## Create model pipeline
model = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1200)),
])

## 5 fold cv
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=32)
scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="recall")
print("CV recall:", scores.mean(), "Standard Deviation:", scores.std())

## Fit model and test it
model.fit(X_train, y_train)
pred = model.predict(X_test)
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred))

CV recall: 0.7426437241563292 Standard Deviation: 0.008340722033274192
[[10727  2870]
 [  140   412]]
              precision    recall  f1-score   support

           0       0.99      0.79      0.88     13597
           1       0.13      0.75      0.21       552

    accuracy                           0.79     14149
   macro avg       0.56      0.77      0.55     14149
weighted avg       0.95      0.79      0.85     14149

